# 06 - Candidate segmentations (DS, Day 9)

The DA plan's Day 9 task for DS: **shortlist 2 to 3 candidate segmentations and share the labels with the DA team.**

The baseline (notebook 05) found two things this notebook acts on:
1. **One cluster (5.6%) was not a customer type.** It held customers with no web sessions: the `has_sessions` / `has_views` flags and the "zero sessions" counts were enough to pull them together. Here, customers with no browsing data get the *typical* browsing values and no flag, so they are placed by what we do know about them (profile and enquiries).
2. **One-hot categories carry a lot of weight in k-means** (84 of 144 columns). Here, each categorical variable is weighted so it counts about as much as one numeric feature.

**Candidates**

| | Model | Why |
|---|---|---|
| A | k-means on the fixed, re-weighted matrix | The baseline, corrected |
| B | Gaussian mixture on principal components | Allows clusters of different shapes and sizes |
| C | k-prototypes (fitted on a sample, then applied to everyone) | Handles numbers and categories natively |
| D | Two-stage k-means: split by intent (Buy / Rent), then cluster each group | Buying and renting are different markets, so they should not share clusters |

**Rules:** no answer key (`_restricted/`) and no outcomes. Candidates are compared on stability, separation, agreement with each other, and whether the clusters make business sense. The README's ARI target is checked only after modelling is final.

## 0. Setup

In [ ]:
%pip install -q scikit-learn

In [ ]:
%pip install -q kmodes

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from pandas.api.types import is_numeric_dtype

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent
FEAT = ROOT / "data" / "features"
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(exist_ok=True)
for f in ["customer_features_mixed.parquet", "customer_features.parquet", "labels_kmeans_baseline.parquet"]:
    assert (FEAT / f).exists(), f"{f} missing: run notebooks 04 and 05 first"

X = pd.read_parquet(FEAT / "customer_features_mixed.parquet").set_index("customer_id")
F = pd.read_parquet(FEAT / "customer_features.parquet").set_index("customer_id").loc[X.index]
base = pd.read_parquet(FEAT / "labels_kmeans_baseline.parquet").set_index("customer_id").iloc[:, 0].loc[X.index]

cat_cols = [c for c in X.columns if not is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if c not in cat_cols]
K = 8                                   # CONFIG: number of clusters (see notebook 05 for why 8)
print(X.shape, f"| {len(num_cols)} numeric, {len(cat_cols)} categorical | K = {K}")

## 1. Fix the data-availability problem
About 5% of customers have no identified web session, so we know nothing about their browsing. Saying "zero sessions" is not the same as "unknown".
- `has_sessions` and `has_views` are removed from the model.
- For those customers, session and view counts, pages and duration take the **median of the customers who do have data**. (The shares were already filled with typical values in notebook 04.)
- They are therefore clustered on profile and enquiry features only.

In [ ]:
no_sess = F["has_sessions"].eq(0)
no_view = F["has_views"].eq(0)
print(f"customers with no sessions: {no_sess.sum()} ({no_sess.mean():.1%}) | no real views: {no_view.sum()} ({no_view.mean():.1%})")

Xn = X[num_cols].drop(columns=["has_sessions", "has_views"], errors="ignore").copy()
for c in ["n_sessions", "mean_pages", "mean_duration_min"]:
    if c in Xn:
        Xn.loc[no_sess, c] = Xn.loc[~no_sess, c].median()
if "n_views" in Xn:
    Xn.loc[no_view, "n_views"] = Xn.loc[~no_view, "n_views"].median()

W_CAT = 0.7071      # a mismatch on a one-hot variable contributes 1 to the squared distance, like a 1 SD gap on a numeric feature
Xc = pd.get_dummies(X[cat_cols], dtype=float) * W_CAT
M = pd.concat([Xn, Xc], axis=1)
print(f"Model matrix: {M.shape[1]} columns ({Xn.shape[1]} numeric + {Xc.shape[1]} weighted category columns)")

## 2. Candidate A: k-means, fixed
Same method as the baseline, on the corrected and re-weighted matrix.

In [ ]:
t0 = time.time()
km = KMeans(n_clusters=K, n_init=20, random_state=0).fit(M)
labA = pd.Series(km.labels_, index=M.index, name="A_kmeans_fixed")
print(f"A fitted in {time.time()-t0:.0f}s | inertia {km.inertia_:,.0f}")
labA.value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("share_%")

## 3. Candidate B: Gaussian mixture on principal components
A mixture model fits clusters that can differ in size and spread. It runs on the first principal components (the directions that carry most of the variation) to keep it fast and stable, then each customer goes to its most likely cluster.

In [ ]:
t0 = time.time()
pca = PCA(n_components=0.90, random_state=0).fit(M)
Z = pca.transform(M)
print(f"PCA keeps {Z.shape[1]} components for {pca.explained_variance_ratio_.sum():.0%} of the variance")
gm = GaussianMixture(n_components=K, covariance_type="full", n_init=2, max_iter=200, random_state=0).fit(Z)
labB = pd.Series(gm.predict(Z), index=M.index, name="B_gmm")
print(f"B fitted in {time.time()-t0:.0f}s | converged: {gm.converged_}")
labB.value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("share_%")

## 4. Candidate C: k-prototypes (optional, slower)
k-prototypes uses distance for the numbers and *matching* for the categories, so no weighting trick is needed. It is slow, so it is **fitted on a random sample** and then applied to every customer.
Set `RUN_KPROTOTYPES = False` to skip it. If the package is missing, the cell says so and carries on.

In [ ]:
RUN_KPROTOTYPES = True
SAMPLE_N = 15_000
labC = None
if RUN_KPROTOTYPES:
    try:
        from kmodes.kprototypes import KPrototypes
        t0 = time.time()
        data = pd.concat([Xn, X[cat_cols].astype(str)], axis=1)
        cat_idx = list(range(Xn.shape[1], data.shape[1]))
        arr = data.to_numpy(dtype=object)
        rng = np.random.default_rng(0)
        sample = rng.choice(len(arr), size=min(SAMPLE_N, len(arr)), replace=False)
        kp = KPrototypes(n_clusters=K, init="Cao", n_init=1, max_iter=15, random_state=0)
        kp.fit(arr[sample], categorical=cat_idx)
        print(f"C fitted on {len(sample)} customers in {time.time()-t0:.0f}s; now assigning everyone ...")
        pred = np.concatenate([kp.predict(arr[i:i + 10_000], categorical=cat_idx) for i in range(0, len(arr), 10_000)])
        labC = pd.Series(pred, index=M.index, name="C_kprototypes")
        print(f"C done in {time.time()-t0:.0f}s")
    except ImportError:
        print("kmodes is not installed, so candidate C is skipped. Run:  %pip install kmodes")
labC.value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("share_%") if labC is not None else "skipped"

## 4b. Candidate D: two-stage k-means (split by intent first)
Notebook 05 showed renters forming their own clean cluster, but candidate A (with categories down-weighted) lets renters and buyers mix. Intent changes the whole picture: rent is a monthly figure, a purchase is a lump sum, and the listings viewed differ.
So D first splits customers by `primary_intent`, then runs k-means separately inside each group. `K_RENT` clusters go to renters and the rest to buyers.

In [ ]:
K_RENT = 2                      # CONFIG: renters are about 21% of customers, so 2 clusters is a starting point
K_BUY = K - K_RENT

def two_stage(seed, n_init):
    lab = pd.Series(index=M.index, dtype=object)
    for intent, k, tag in [("Rent", K_RENT, "R"), ("Buy", K_BUY, "B")]:
        idx = X.index[X["primary_intent"].eq(intent)]
        part = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(M.loc[idx])
        lab.loc[idx] = [f"{tag}{c}" for c in part]
    return lab

t0 = time.time()
labD = two_stage(0, 10).rename("D_two_stage")
print(f"D fitted in {time.time()-t0:.0f}s | renters: {int(X['primary_intent'].eq('Rent').sum())}, buyers: {int(X['primary_intent'].eq('Buy').sum())}, other: {int((~X['primary_intent'].isin(['Rent','Buy'])).sum())}")
assert labD.notna().all(), "Some customers have an intent other than Buy or Rent: handle them before clustering"
labD.value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("share_%")

## 5. Compare the candidates
Four views, none of which uses the answer key:
- **Size:** the smallest cluster. Tiny clusters are usually noise.
- **Separation:** silhouette on a 15,000-customer sample, in the same matrix for every candidate (low values are normal here).
- **Stability:** re-fit with different random starts and compare with the main run (adjusted Rand index; 1 = identical).
- **Agreement:** do different methods find the same groups? (ARI between candidates). High agreement means the structure is real, not an artefact of one method.

In [ ]:
cands = {"baseline (05)": base, "A": labA, "B": labB}
if labC is not None:
    cands["C"] = labC
cands["D"] = labD
cands = {k: v.rename(k) for k, v in cands.items()}

sil_idx = np.random.default_rng(0).choice(len(M), size=min(15_000, len(M)), replace=False)
rows = []
for name, lab in cands.items():
    sh = lab.value_counts(normalize=True) * 100
    rows.append({"candidate": name, "clusters": lab.nunique(), "smallest_%": round(sh.min(), 1), "largest_%": round(sh.max(), 1),
                 "silhouette": round(silhouette_score(M.iloc[sil_idx], lab.iloc[sil_idx]), 3)})
summary = pd.DataFrame(rows).set_index("candidate")

stab = {}
stab["A"] = np.mean([adjusted_rand_score(labA, KMeans(n_clusters=K, n_init=3, random_state=s).fit_predict(M)) for s in range(1, 6)])
stab["B"] = np.mean([adjusted_rand_score(labB, GaussianMixture(n_components=K, covariance_type="full", n_init=1, max_iter=200, random_state=s).fit(Z).predict(Z)) for s in range(1, 5)])
stab["D"] = np.mean([adjusted_rand_score(labD, two_stage(s_, 3)) for s_ in range(1, 4)])
summary["stability_ARI"] = pd.Series(stab).round(3)
display(summary)

agree = pd.DataFrame({a: {b: adjusted_rand_score(cands[a], cands[b]) for b in cands} for a in cands}).round(2)
print("Agreement between candidates (ARI):")
display(agree)

## 6. What is in each candidate?
Plain-unit profiles for every candidate, using the readable features. Read them next to each other: the same customer type should show up in each, and each cluster should be explainable in a sentence.

In [ ]:
def profile(lab):
    p = pd.DataFrame(index=sorted(lab.unique()))
    p["customers"] = lab.value_counts().sort_index()
    p["share_%"] = (p["customers"] / len(lab) * 100).round(1)
    p["buy_%"] = (F["primary_intent"].eq("Buy").groupby(lab).mean() * 100).round(0)
    p["overseas_%"] = (F["is_overseas"].astype(float).groupby(lab).mean() * 100).round(0)
    p["median_budget"] = F["budget_gbp"].groupby(lab).median().round(0)
    p["budget_missing_%"] = (F["budget_missing"].groupby(lab).mean() * 100).round(0)
    for c, n in [("share_mobile", "mobile_%"), ("share_new_build", "new_build_%"), ("share_offplan", "offplan_%")]:
        if c in F.columns:
            p[n] = (F[c].groupby(lab).mean() * 100).round(0)
    p["median_sessions"] = F["n_sessions"].groupby(lab).median()
    p["no_sessions_%"] = ((F["has_sessions"] == 0).groupby(lab).mean() * 100).round(0)
    p["top_buyer_position"] = F["buyer_position"].fillna("Unknown").groupby(lab).agg(lambda s: s.value_counts().index[0])
    p["top_finance"] = F["finance_type"].fillna("Unknown").groupby(lab).agg(lambda s: s.value_counts().index[0])
    p["top_lead_source"] = F["primary_lead_source"].fillna("No lead").groupby(lab).agg(lambda s: s.value_counts().index[0])
    return p

for name, lab in cands.items():
    print(f"\n=== {name} ===")
    display(profile(lab))

## 7. Shortlist and save
Choose 2 or 3 to share with the DA team. Decide on **stability, size, agreement and whether each cluster makes sense**, and write the reason down.
The DA team will profile the shortlisted candidates on Day 10-11, including against outcomes, which this notebook never touches.

In [ ]:
SHORTLIST = [k for k in cands if k != "baseline (05)"]      # CONFIG: trim to the 2 or 3 you choose after reading sections 5 and 6

out = pd.concat([cands[k] for k in cands], axis=1)
out.reset_index().to_parquet(FEAT / "labels_candidates.parquet", index=False)
summary.to_csv(OUT / "candidate_comparison.csv")
agree.to_csv(OUT / "candidate_agreement.csv")
for k in SHORTLIST:
    profile(cands[k]).to_csv(OUT / f"candidate_{k}_profile.csv")
print("Saved labels for", list(cands), "->", FEAT / "labels_candidates.parquet")
print("Shortlist:", SHORTLIST)

## 8. Validate

In [ ]:
results = {
    "every customer has a label in every candidate": bool(out.notna().all().all()) and len(out) == len(X),
    "no candidate has a cluster under 1% of customers": bool((summary.loc[[c for c in summary.index if c != 'baseline (05)'], 'smallest_%'] >= 1).all()),
    "no cluster is defined by missing browsing data (A)": bool(profile(labA)["no_sessions_%"].max() < 50),
    "candidate A is stable (ARI >= 0.7)": bool(stab["A"] >= 0.7),
    "answer key never read": True,
}
for k_, v in results.items():
    print(("PASS  " if v else "CHECK ") + k_)
print("\n'CHECK' is something to look at and write down, not an error.")

## Hand-off notes
- **Share with DA (Day 9):** `data/features/labels_candidates.parquet` and `outputs/candidate_comparison.csv`, plus the profile CSVs for the shortlist.
- **Write down:** which candidates you shortlisted and why, the stability numbers, and the baseline's no-sessions cluster and how this notebook handled it.
- **Day 10:** stability checks on the final choice (bootstrap, not just random starts), then write `customer_segments` with `model_version = seg-v1.0`.
- **Only when the model is final:** run `recoverability_check.py` once against the answer key, and report the ARI as a final result. Do not tune against it.